   ... for Q3/Q4-style tasks... do NOT start by chasing every functiion...
   Start from the task contract and the pseudocode... Then inspect only enough 
   code to answer "what inputs do I get, what state do I update, what helper
   already exists"...

   A good method:

1. Read the task statement twice:
   Extract only:
   - function to edit
   - files allowed to edit
   - exact input/output behaviour
   - pseudocode
   - tests to run

2. Find the function body
   ... `ctrl + shift + F`

3. Look at the function signature first.
   ... this tells you almost everything...    #
```
deptname    -> current department name
member      -> current person
state       -> hidden pointer, probably `ageinfo *`
```
   Then the first line confirms it... 

   So the task is just: update `a` using `deptname` and `member`.

4. Trust pseudocode unless it conflicts with tests.
   For  T3, spec literally says... 
   ... do not over-investigate the whole project before implementing...

5. For callbacks, draw a 3-line call cabin...
   For T4 in `dlists`:
```
dlist_print()
    -> dlist_foreach_member(dl, callback, out)
        -> callback(deptname, member, out)
```
   ... that's enough, you don't need to deeply understand every nested wrapper...

6. When you see `void *state`, ask one question...
   `What real type is being hidden inside this void * (generic pointer)`

   EXAMPLES:
```c
FILE *out = (FILE *)state;
ageinfo *a = (ageinfo *)state;
int *sum = (int *)state;
```
   `void *state` is just a way to smuggle extra context into a callback...

---
7. Use source chasing selectively:
   Yes, `C+S+F`/"go to definition" is useful, but only for these things:
   - What does this helper return??
   - Does this helper own/copy/free memory?
   - What callback signature does this iterator expect?
   - What format does this print function already produce?

   Do not chase every call recursively... 


---

   .... conceptual break down of how Iterators and Callbacks work in C, followed
   ...

---
THE CONCEPT: Iterators, Callbacks, and the `void *state`
   In modern languages like C++ or Python, if you want to loop over a complex
   data structure (like a BST or a Hash Map), you use a `for-each` loop. The
   language magically handles the traversal for you. 

   C does not have this magic. If a library provides a complex `list` or `bst`
   (like in your `dlists` or ACIS...) the library author doesn't want you
   touching their internal `struct node` pointers, because you might accidentally
   break the tree,

   ... Instead, they use the ITERATOR + CALLBACK pattern.

   Think of it like a TOUR GUIDE (The Iterator) and a TOURIST (the Callback):
   1. THE ITERATOR (The Tour Guide): A function built into the library. It knows
      exactly how to navigate the complex maze of the data structure. It walks
      through every item, one by one.
   2. THE CALLBACK (The Tourist): A function you write. You give it to the
      ... every time the iterator stops at a new item, it triggers your Callback
      so you can look at the item and do something with it.
      
---
THE PROBLEM: MEMORY AMNESIA
   When your callback fires, it only knows about the current item. If you are
   trying to calculate a running total, or print items to a specific file, your
   Callback has a problem: it forgets everything between steps...

THE SOLUTION:
   To fix this, C iterators allow you to pass a (generic pointer) `void *state`
   variable.
   Think of this as a generic BACKPACK...

   1. You pack the backpack with whatever you need (a pointer to an `int` for a
      total, or a `FILE *` for writing...)
   2. You hand the backpack (`void *state`)  to the Iterator...
   3. The iterator doesn't look inside, it jus blindly hands the backpack
      `void *state`  to your Callback every single time it fires.
   4. Your callback opens the backpack `*state` back to the real type... and 
      uses the data inside...

```c
static void my_callback(ItemType item, void *state) {
    int backpack = *(int *)state
    backpack += item;
}

int total = 0;

list_foreach(my_list, my_callback, &total);
```



```c
typedef void (*float_cb)(float val, void *state);
void foreach_float(float *arr, int n, float_cb cb, void *state);
```

```c
#include <stdio.h>

#define NELEMENTS(arr) (sizeof(arr) / sizeof(arr[0]))

static void sum_floats_cb(float val, void *state) {
    float total = *(float *)state;
    total += val;
}

float sum_floats() {
    float prices[] = {1.50f, 2.00f, 3.25f};
    float total = 0.0f;

    foreach_float(prices, NELEMENTS(prices), sum_floats_cb, &total);
    printf("Total is: %d", total);
}
```

---

```c
typedef void (*name_cb)(char *name, void *state);
void foreach_name(char **names, int n, name_cb cb, void *state);
```

```c
static void print_name_cb(char* name, void *state) {
    FILE *out = (FILE *)state;
    printf(out, "%s\n", name);
}

void display_names(char **names, int n) {
    foreach_name(names, n, print_name_cb, stdout);
}
```

```c
typedef void (*line_cb)(int line_num, char *line_text);
void foreveryline(char *filename, line_cb cb);
```

```c
#include <stdio.h>
#include <string.h>

static int error_count = 0;

void error_detector_cb(line_num, char *line_text) {
    if (strstr(line_text, "ERROR") != NULL) {
        error_count++;
    }
}

void analyze_log(char *filename) {
    error_count = 0;
    foreveryline(filename, error_detector_cb);
    printf("Found %d errors inside %s\n", error_count, filename);
}
```


---


```c
#include <assert.h>

typedef struct { 
    char name[32]; 
    int age; 
} Person;

typedef void (*generic_cb)(void *element, void *state);
void list_foreach(List *l, generic_cb cb, void *state);

static void find_oldest(void *element, void *state) {
    Person *p = (Person *)element;
    int *max_age = (int *)state;

    if (p->age > *max_age) {
        *max_age = p->age;
    }
}

void find_oldest_age(List *l) {
    assert(l != NULL);
    int max = -1;
    list_for_each(l, check_age_cb, &max);
    printf("The oldest age is %d\n", max);
}
```


```c

// Assume set_size() returns the number of items in a set
int set_size(Set *s);

// The callback provides the Key, the Value (as void*), and State
typedef void (*bst_kv_cb)(char *key, void *value, void *state);
void foreach_bst(BST *tree, bst_kv_cb cb, void *state);

static void print_include_counts(char *key, void *value, void *state) {
    Set *includes = (Set *)value;
    FILE *out = (FILE *)state;

    fprintf(out, "%S includes %d files\n", key, set_size(includes));
}

void display_analysis(BST *root) {
    foreach_bst(root, print_include_counts, stdout);
}
```


---


```c
void list_add(List *l, void *element); // Adds an element to a list
void list_foreach(List *l, generic_cb cb, void *state);

static void(filter_adults(void *element, void *state)) {
    Person *p = (Person *)element;
    List *adult_list = (List *)state;       // Our backpack is the new list!

    if (p->age > 18) {
        list_add(adult_list, p);            // Add them to the new list
    }
}

List *get_all_adults(List *all_people) {
    List *adults = make_list();             // 1. Create the new list (the backpack)

    list_foreach(all_people, filter_adults, adults);    // 2. Iterate, passing the new list as the state

    return adults;                                      // 3. Return the fully populated list...
}
```

---

-- A CALLBACK is a function passed into another function as an argument, which 
   is then executed (or "called back") by that outer function to complete a 
   specific tassk or handle an event. It essentially teslls the program, "Do
   this task, and when you're finished, run this other function..."

   COMMON USE CASES
   - ARRAY/LIST ITERATION: Customising how a program processes every single
     item in a list. 

---

1. Find the callback typedef.
2. Match your function signature exactly
3. Decide what each parameter really means
4. Cast void *state back to the real type
5. Let the iterator do the loop
6. Put only one-item logic inside the callback...